In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from datetime import datetime

# Lecture Bronze
df = spark.table("pharma_catalog.bronze.bronze_warehouses")
display(df)

In [0]:
# COMMAND ----------
# Doublons
df_dedup = df.dropDuplicates(["warehouse_id"])
doublons = df.count() - df_dedup.count()
print(f"🔁 Doublons détectés : {doublons}")

# COMMAND ----------
# Normalisation country en majuscules
df_normalized = df_dedup.withColumn("country", F.upper(F.col("country")))

df_quality = df_normalized.withColumn(
    "rule_failed",
    F.when(F.col("warehouse_id").isNull(), "null_warehouse_id")
    .when(F.col("warehouse_name").isNull(), "null_warehouse_name")
    .when(F.col("country").isNull(), "null_country")
    .when(F.col("city").isNull(), "null_city")
    .when(F.col("warehouse_type").isNull(), "null_warehouse_type")
    .when(F.col("capacity_units").isNull(), "null_capacity_units")
    .when(F.col("capacity_units") < 0, "capacity_negative")
    .when(F.col("manager_name").isNull(), "null_manager_name")
    .otherwise(None)
)

passed = df_quality.filter(F.col("rule_failed").isNull()).drop("rule_failed")
failed = df_quality.filter(F.col("rule_failed").isNotNull())

print(f" Lignes OK : {passed.count()}")
print(f"Lignes rejetées : {failed.count()}")

In [0]:
# COMMAND ----------
display(failed)

In [0]:
# COMMAND ----------
if failed.count() > 0:
    df_quarantine = failed.withColumn("source_table", F.lit("bronze_warehouses")) \
                           .withColumn("processing_date", F.lit(datetime.now().strftime("%Y-%m-%d")))

    df_quarantine.write.mode("append") \
                       .option("mergeSchema", "true") \
                       .saveAsTable("pharma_catalog.silver_quarantine.quarantine")
    print(f"✅ {failed.count()} ligne(s) envoyée(s) en quarantine")
else:
    print("✅ Aucune ligne rejetée — quarantine vide")

In [0]:
# COMMAND ----------
# Envoi des lignes propres vers Silver
passed.write.mode("overwrite").saveAsTable("pharma_catalog.silver.silver_warehouses")

print(f"✅ {passed.count()} ligne(s) chargée(s) dans silver_warehouses")

In [0]:
# COMMAND ----------
display(spark.table("pharma_catalog.silver.silver_warehouses"))

# COMMAND ----------
display(spark.table("pharma_catalog.silver_quarantine.quarantine"))